In [100]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import psycopg2

In [101]:
connection = psycopg2.connect(user = "postgres",
                              password = 'itmm1315@VZU', 
                              host = 'localhost',
                              port = '5432',
                              database = 'patient_db')

In [102]:
cursor = connection.cursor()
cursor.execute('Select * from patients')

In [103]:
results = cursor.fetchall()

In [104]:
df = pd.read_sql_query('Select * from patients', connection)

C:\Users\awneesh.kumar\AppData\Local\Temp\ipykernel_18580\590897752.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query('Select * from patients', connection)


In [105]:
df.describe()

,age,weight,height,income_lpa
count,100.000000,100.000000,100.000000,100.000000
mean,47.180000,83.894000,1.713200,18.400700
std,16.649312,21.020278,0.110205,16.067515
min,18.000000,51.100000,1.500000,0.530000
25%,34.750000,63.650000,1.610000,2.897500
50%,47.000000,82.300000,1.730000,14.125000
75%,61.000000,101.300000,1.810000,30.162500
max,75.000000,119.800000,1.900000,50.000000


In [106]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   age                         100 non-null    int64  
 1   weight                      100 non-null    float64
 2   height                      100 non-null    float64
 3   income_lpa                  100 non-null    float64
 4   smoker                      100 non-null    bool   
 5   city                        100 non-null    object 
 6   occupation                  100 non-null    object 
 7   insurance_premium_category  100 non-null    object 
dtypes: bool(1), float64(3), int64(1), object(3)
memory usage: 5.7+ KB


In [107]:
df.isnull().sum()

age                           0
weight                        0
height                        0
income_lpa                    0
smoker                        0
city                          0
occupation                    0
insurance_premium_category    0
dtype: int64

In [108]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
4,69,62.2,1.60,3.94,True,Indore,retired,High
27,58,111.4,1.78,34.33,False,Lucknow,private_job,Medium
56,24,101.9,1.55,2.86,True,Kolkata,student,Medium
92,37,62.7,1.85,30.00,True,Lucknow,government_job,Low
59,61,118.7,1.82,1.13,False,Jaipur,retired,High


In [109]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [110]:
df['city'].unique()

array(['Jaipur', 'Chennai', 'Indore', 'Mumbai', 'Kota', 'Hyderabad',
       'Delhi', 'Chandigarh', 'Pune', 'Kolkata', 'Lucknow', 'Gaya',
       'Jalandhar', 'Mysore', 'Bangalore'], dtype=object)

Define feature 1 as bmi

In [111]:
df_feat = df.copy()
df_feat['bmi'] = df_feat['weight']/(df_feat['height']**2)

Define feature 2 as age_group

In [112]:
def age_group(age):
    if age < 28:
        return 'Young'
    elif age < 45:
        return 'Adult'
    elif age <65:
        return 'Middle_aged'
    return 'Senior'

In [113]:
df_feat['age_group'] = df_feat['age'].apply(age_group)

Feature 3: Lifestyle risk

In [114]:
def lifestyle_risk(row):
    if row['smoker'] and row['bmi'] > 30:
        return "High"
    elif row['smoker'] or row['bmi'] > 25:
        return 'Medium'
    else:
        return 'Low'

In [115]:
df_feat['lifestyle_risk'] = df_feat.apply(lifestyle_risk, axis=1)
df_feat

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk
0,67,119.8,1.56,2.92,False,Jaipur,retired,High,49.227482,Senior,Medium
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low,30.189017,Adult,Medium
2,39,56.8,1.64,36.64,False,Indore,freelancer,Low,21.118382,Adult,Low
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium,45.535900,Young,High
4,69,62.2,1.60,3.94,True,Indore,retired,High,24.296875,Senior,Medium
...,...,...,...,...,...,...,...,...,...,...,...
95,36,52.8,1.57,19.64,False,Indore,business_owner,Low,21.420747,Adult,Low
96,26,113.8,1.54,34.01,False,Delhi,private_job,Low,47.984483,Young,Medium
97,52,60.8,1.80,44.86,False,Hyderabad,freelancer,Low,18.765432,Middle_aged,Low
98,27,101.1,1.82,28.30,False,Kolkata,business_owner,Low,30.521676,Young,Medium


In [116]:
class1_cities = ["Mumbai", "Delhi", "Jaipur", "Bangalore", "Chandigarh", "Chennai", "Kolkata", "Lucknow", "Hyderabad", "Pune"]
class2_cities = [
    "Indore", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

Feature 4 as City Tier

In [117]:
def city_class(city):
    if city in class1_cities:
        return 1
    elif city in class2_cities:
        return 2
    else:
        return 3

In [118]:
df_feat['city_class'] = df_feat['city'].apply(city_class)

Drop all other columns from the features now

In [119]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])

,income_lpa,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk,city_class
0,2.92,retired,High,49.227482,Senior,Medium,1
1,34.28,freelancer,Low,30.189017,Adult,Medium,1
2,36.64,freelancer,Low,21.118382,Adult,Low,2
3,3.34,student,Medium,45.535900,Young,High,1
4,3.94,retired,High,24.296875,Senior,Medium,2
...,...,...,...,...,...,...,...
95,19.64,business_owner,Low,21.420747,Adult,Low,2
96,34.01,private_job,Low,47.984483,Young,Medium,1
97,44.86,freelancer,Low,18.765432,Middle_aged,Low,1
98,28.30,business_owner,Low,30.521676,Young,Medium,1


In [120]:
X = df_feat[['bmi', 'age_group', 'lifestyle_risk', 'city_class','income_lpa', 'occupation']]
y = df_feat[['insurance_premium_category']]
y

,insurance_premium_category
0,High
1,Low
2,Low
3,Medium
4,High
...,...
95,Low
96,Low
97,Low
98,Low


Define categorical and numerical features

In [121]:
categorical_features = ['age_group', 'lifestyle_risk', 'occupation', 'city_class']
numeric_features = ['bmi', 'income_lpa']

Create column transformer for OHE

In [122]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(), categorical_features), 
                                               ('num', 'passthrough', numeric_features)])

Create a pipeline with preprocessing and random forest classifier. 

In [123]:
from sklearn.ensemble import RandomForestClassifier
pipeline_rf = Pipeline(steps=[('preprocessor', preprocessor), 
                           ('classifier', RandomForestClassifier(random_state=42))])

In [124]:
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ]
)

pipeline_svm = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LinearSVC(random_state=42))
])

#Spiliting data for training and testing

In [125]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline_rf.fit(X_train, y_train)
pipeline_svm.fit(X_train, y_train)

c:\Users\awneesh.kumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\awneesh.kumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

Predict and evaluate 

In [126]:
from sklearn.metrics import accuracy_score 
y_pred = pipeline_rf.predict(X_test)
accuracy_score(y_test, y_pred)

0.55

In [127]:
y_pred_svm = pipeline_svm.predict(X_test)
accuracy_score(y_test, y_pred_svm)

0.75

In [128]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_class,income_lpa,occupation
32,31.495845,Middle_aged,Medium,2,50.00,private_job
33,21.791064,Senior,Low,1,1.46,retired
36,21.713266,Middle_aged,Low,1,0.53,retired
56,42.414152,Young,High,1,2.86,student
78,27.932798,Middle_aged,Medium,2,14.74,freelancer


Save the trained pipeline using pickle

In [130]:
import pickle
pickle_model_path = 'model.pkl'
with open(pickle_model_path, 'wb') as f:
    pickle.dump(pipeline_svm, f)